In [3]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/payments.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [4]:
 
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.


Loaded payments.csv
Rows: 45,000
Columns: 10


,payment_id,order_id,customer_id,payment_date,payment_method,payment_provider,payment_status,transaction_reference,amount,refund_amount
0,PAY-00000001,ORD-00000001,CUS-006832,2025-06-28,Digital Wallet,WalletPay,Successful,TXN-17397934713,479.49,0.0
1,PAY-00000002,ORD-00000002,CUS-002041,2023-12-25,Digital Wallet,WalletPay,Successful,TXN-90267437752,212.90,0.0
2,PAY-00000003,ORD-00000003,CUS-006564,2024-05-10,Cash,Store POS,Successful,TXN-18291880508,292.61,0.0
3,PAY-00000004,ORD-00000004,CUS-002664,2021-08-12,Buy Now Pay Later,FlexPay,Successful,TXN-32399992560,661.53,0.0
4,PAY-00000005,ORD-00000005,CUS-000822,2020-09-16,Card,Bank Gateway,Successful,TXN-71623957412,157.93,0.0


In [5]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,45000
1,columns,10
2,duplicates,0
3,missing_cells,1539


Decision point: determine which findings require remediation.


In [6]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,payment_id,str,45000,0,45000
1,order_id,str,45000,0,45000
2,customer_id,str,43461,1539,11699
3,payment_date,str,45000,0,2434
4,payment_method,str,45000,0,5
5,payment_provider,str,45000,0,15
6,payment_status,str,45000,0,3
7,transaction_reference,str,45000,0,45000
8,amount,float64,45000,0,25980
9,refund_amount,float64,45000,0,2508


In [7]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count
customer_id,1539


In [8]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [9]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


,count,mean,std,min,25%,50%,75%,max
amount,45000.0,274.254778,225.907247,8.62,122.5175,213.27,359.455,3255.34
refund_amount,45000.0,16.620415,85.589017,0.00,0.0000,0.00,0.000,1740.93


,iqr_extreme_rate
refund_amount,0.059511
amount,0.044289


In [10]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,payment_id,45000,0,"{'PAY-00000001': 1, 'PAY-00000002': 1, 'PAY-00..."
1,order_id,45000,0,"{'ORD-00000001': 1, 'ORD-00000002': 1, 'ORD-00..."
2,customer_id,11699,0,"{nan: 1539, 'CUS-007859': 13, 'CUS-001571': 13..."
3,payment_date,2434,0,"{'2026-02-14': 33, '2024-05-24': 32, '2022-11-..."
4,payment_method,5,0,"{'Card': 9116, 'Digital Wallet': 9026, 'Cash':..."
5,payment_provider,15,0,"{'Bank Gateway': 9058, 'WalletPay': 8970, 'Sto..."
6,payment_status,3,0,"{'Successful': 40491, 'Reversed': 2678, 'Pendi..."
7,transaction_reference,45000,0,"{'TXN-17397934713': 1, 'TXN-90267437752': 1, '..."


In [11]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


,column,parse_failures,min,max
0,payment_date,0,2020-01-01,2026-08-30


In [12]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,payment_id,1.000
1,order_id,1.000
7,transaction_reference,1.000
8,amount,0.577
2,customer_id,0.260
9,refund_amount,0.056
3,payment_date,0.054
4,payment_method,0.000
5,payment_provider,0.000
6,payment_status,0.000


In [14]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if isinstance(numeric_pairs, pd.DataFrame) and not numeric_pairs.empty:
    display(
        numeric_pairs.sort_values(
            "correlation",
            key=lambda s: s.abs(),
            ascending=False
        ).head(20)
    )


,level_0,level_1,correlation
0,amount,amount,1.000000
3,refund_amount,refund_amount,1.000000
1,amount,refund_amount,0.157402
2,refund_amount,amount,0.157402


In [15]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


,issue,column,count
0,missing_values,customer_id,1539
1,duplicate_rows,NaN,0


Consulting decision: validate material findings against business rules before cleaning.


In [16]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
audit_summary.to_csv(f"outputs/{stem}_audit_summary.csv",index=False)  # Save the audit summary for downstream review.


,dataset,rows,columns,duplicates,missing_cells
0,payments.csv,45000,10,0,1539
